Design a spell correction model. spell-errors.txt file consists of a correct word and the 
possible errors such as raining: rainning, raning. You are provided with the 
count_1edit.txt file that consists of possible errors in 1 edit distance. Similarly use the 
counts or frequencies of the words in count_unigram.txt file. Use a noisy channel model 
for finding the correct word from an incorrect word. Find the accuracy of your spell 
checker. [You will find all files in the Spell-Correction folder.]

In [1]:
from collections import Counter, defaultdict
from pathlib import Path
from string import ascii_lowercase
import csv
import json
import math
import re

LAB_DIR = Path.cwd() if Path.cwd().name.lower() == "lab 8" else Path.cwd() / "Lab 8"
DATA_DIR = LAB_DIR / "Spell-Correction"
OUTPUT_DIR = LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALPHA = 0.1


def load_unigrams(path):
    counts = {}
    with path.open(encoding="utf-8") as data_file:
        for line_number, line in enumerate(data_file, start=1):
            fields = line.rstrip("\r\n").rsplit("\t", 1)
            if len(fields) != 2:
                raise ValueError(f"Invalid unigram row at line {line_number}")
            word, count = fields
            word = word.casefold()
            if word.isalpha():
                counts[word] = counts.get(word, 0) + int(count)
    if not counts:
        raise ValueError("The unigram file contains no alphabetic words")
    return counts


def load_edit_counts(path):
    counts = Counter()
    with path.open(encoding="cp1252") as data_file:
        for line_number, line in enumerate(data_file, start=1):
            fields = line.rstrip("\r\n").split("\t")
            if len(fields) != 2 or "|" not in fields[0]:
                raise ValueError(f"Invalid edit-count row at line {line_number}")
            correct_piece, error_piece = fields[0].split("|", 1)
            counts[(correct_piece, error_piece)] += int(fields[1])
    if not counts:
        raise ValueError("The edit-count file contains no edit patterns")
    return counts


def load_spelling_pairs(path):
    pairs = []
    with path.open(encoding="utf-8") as data_file:
        for line_number, line in enumerate(data_file, start=1):
            if ":" not in line:
                raise ValueError(f"Invalid spelling-error row at line {line_number}")
            correct_word, error_list = line.split(":", 1)
            correct_word = correct_word.strip().casefold()
            for item in error_list.split(","):
                item = item.strip()
                if not item:
                    continue
                match = re.fullmatch(r"(.+?)(?:\*(\d+))?", item)
                if match is None:
                    raise ValueError(f"Invalid spelling variant at line {line_number}: {item}")
                error_word = match.group(1).strip().casefold()
                frequency = int(match.group(2) or 1)
                if correct_word and error_word and frequency > 0:
                    pairs.append({"correct": correct_word, "error": error_word, "frequency": frequency})
    if not pairs:
        raise ValueError("The spelling-error file contains no correction pairs")
    return pairs


unigram_counts = load_unigrams(DATA_DIR / "count_unigram.txt")
edit_counts = load_edit_counts(DATA_DIR / "count_1edit.txt")
spelling_pairs = load_spelling_pairs(DATA_DIR / "spell-errors.txt")
total_unigram_count = sum(unigram_counts.values())
print(f"Loaded {len(unigram_counts):,} dictionary words and {len(spelling_pairs):,} weighted spelling pairs.")


Loaded 333,333 dictionary words and 39,710 weighted spelling pairs.


In [2]:
def candidates_one_edit(error_word, vocabulary):
    candidates = set()
    for index in range(len(error_word)):
        candidates.add(error_word[:index] + error_word[index + 1:])
        if index + 1 < len(error_word):
            swapped = list(error_word)
            swapped[index], swapped[index + 1] = swapped[index + 1], swapped[index]
            candidates.add("".join(swapped))
        for letter in ascii_lowercase:
            if letter != error_word[index]:
                candidates.add(error_word[:index] + letter + error_word[index + 1:])
    for index in range(len(error_word) + 1):
        for letter in ascii_lowercase:
            candidates.add(error_word[:index] + letter + error_word[index:])
    return sorted(candidate for candidate in candidates if candidate != error_word and candidate in vocabulary)


def edit_signatures(correct_word, error_word):
    signatures = set()
    if abs(len(correct_word) - len(error_word)) > 1:
        return signatures
    if len(correct_word) == len(error_word):
        differences = [index for index, (left, right) in enumerate(zip(correct_word, error_word)) if left != right]
        if len(differences) == 1:
            index = differences[0]
            signatures.add((correct_word[index], error_word[index]))
        elif (
            len(differences) == 2
            and differences[1] == differences[0] + 1
            and correct_word[differences[0]] == error_word[differences[1]]
            and correct_word[differences[1]] == error_word[differences[0]]
        ):
            start = differences[0]
            signatures.add((correct_word[start:start + 2], error_word[start:start + 2]))
        return signatures

    prefix = 0
    while prefix < min(len(correct_word), len(error_word)) and correct_word[prefix] == error_word[prefix]:
        prefix += 1
    suffix = 0
    while (
        suffix < min(len(correct_word) - prefix, len(error_word) - prefix)
        and correct_word[len(correct_word) - suffix - 1] == error_word[len(error_word) - suffix - 1]
    ):
        suffix += 1
    correct_end = len(correct_word) - suffix if suffix else len(correct_word)
    error_end = len(error_word) - suffix if suffix else len(error_word)
    correct_piece = correct_word[prefix:correct_end]
    error_piece = error_word[prefix:error_end]

    if prefix:
        correct_piece = correct_word[prefix - 1] + correct_piece
        error_piece = error_word[prefix - 1] + error_piece
    elif suffix:
        correct_piece += correct_word[correct_end]
        error_piece += error_word[error_end]
    signatures.add((correct_piece, error_piece))
    return signatures



In [3]:
edit_totals = Counter()
edit_outcomes = defaultdict(set)
inserted_character_counts = Counter()
deleted_character_counts = Counter()
substitution_counts = Counter()
for (error_piece, correct_piece), count in edit_counts.items():
    edit_totals[correct_piece] += count
    edit_outcomes[correct_piece].add(error_piece)
    if len(error_piece) == len(correct_piece) + 1:
        for index in range(len(error_piece)):
            if correct_piece == error_piece[:index] + error_piece[index + 1:]:
                inserted_character_counts[error_piece[index]] += count
                break
    elif len(correct_piece) == len(error_piece) + 1:
        for index in range(len(correct_piece)):
            if error_piece == correct_piece[:index] + correct_piece[index + 1:]:
                deleted_character_counts[correct_piece[index]] += count
                break
    elif len(correct_piece) == len(error_piece) == 1 and correct_piece != error_piece:
        substitution_counts[correct_piece] += count

global_edit_total = sum(edit_counts.values())
global_edit_types = len(edit_counts)
total_insertions = sum(inserted_character_counts.values())
total_deletions = sum(deleted_character_counts.values())
total_substitutions = sum(substitution_counts.values())


def backoff_probability(counts, total, character):
    return (counts[character] + ALPHA) / (total + ALPHA * (len(counts) + 1))


def channel_probability(correct_word, error_word):
    signatures = edit_signatures(correct_word, error_word)
    if not signatures:
        return 0.0
    probabilities = []
    for correct_piece, error_piece in signatures:
        count = edit_counts[(error_piece, correct_piece)]
        denominator = edit_totals[correct_piece]
        outcome_count = len(edit_outcomes[correct_piece])
        if count:
            probability = (count + ALPHA) / (denominator + ALPHA * (outcome_count + 1))
        elif len(error_piece) == len(correct_piece) + 1:
            inserted = next(
                (error_piece[index] for index in range(len(error_piece))
                 if correct_piece == error_piece[:index] + error_piece[index + 1:]),
                error_piece[-1],
            )
            probability = backoff_probability(inserted_character_counts, total_insertions, inserted)
        elif len(correct_piece) == len(error_piece) + 1:
            deleted = next(
                (correct_piece[index] for index in range(len(correct_piece))
                 if error_piece == correct_piece[:index] + correct_piece[index + 1:]),
                correct_piece[-1],
            )
            probability = backoff_probability(deleted_character_counts, total_deletions, deleted)
        elif len(correct_piece) == 1:
            probability = backoff_probability(substitution_counts, total_substitutions, correct_piece)
        else:
            probability = ALPHA / (global_edit_total + ALPHA * (global_edit_types + 1))
        probabilities.append(probability)
    return max(probabilities)

def noisy_channel_score(correct_word, error_word):
    word_probability = unigram_counts[correct_word] / total_unigram_count
    error_probability = channel_probability(correct_word, error_word)
    if error_probability <= 0:
        return float("-inf")
    return math.log(word_probability) + math.log(error_probability)


def correct_spelling(error_word, vocabulary, candidate_cache):
    if error_word not in candidate_cache:
        candidate_cache[error_word] = candidates_one_edit(error_word, vocabulary)
    candidates = candidate_cache[error_word]
    if not candidates:
        return None, []
    scored = sorted(
        ((candidate, noisy_channel_score(candidate, error_word)) for candidate in candidates),
        key=lambda item: (-item[1], item[0]),
    )
    return scored[0][0], scored




In [4]:
candidate_cache = {}
predictions = []
for pair in spelling_pairs:
    predicted, scored = correct_spelling(pair["error"], unigram_counts, candidate_cache)
    predictions.append({
        **pair,
        "prediction": predicted or "",
        "correct_in_candidates": any(candidate == pair["correct"] for candidate, _ in scored),
        "top_candidates": ";".join(candidate for candidate, _ in scored[:5]),
    })
print(f"Generated candidates for {len(candidate_cache):,} distinct misspellings.")


Generated candidates for 36,978 distinct misspellings.


In [5]:
total_weight = sum(row["frequency"] for row in predictions)
correct_weight = sum(row["frequency"] for row in predictions if row["prediction"] == row["correct"])
covered_weight = sum(row["frequency"] for row in predictions if row["correct_in_candidates"])
covered_correct_weight = sum(
    row["frequency"] for row in predictions
    if row["correct_in_candidates"] and row["prediction"] == row["correct"]
)
summary = {
    "model": "argmax_correct_word(P(error|correct_word) * P(correct_word))",
    "channel": "P(error_piece|correct_piece) from count_1edit.txt with add-alpha smoothing and edit-character backoff, alpha=0.1",
    "candidate_generation": "dictionary words at exactly one Damerau edit",
    "evaluation_pairs": len(predictions),
    "distinct_errors": len(candidate_cache),
    "frequency_weighted_examples": total_weight,
    "frequency_weighted_accuracy": correct_weight / total_weight,
    "candidate_coverage": covered_weight / total_weight,
    "accuracy_when_gold_is_in_candidates": covered_correct_weight / covered_weight if covered_weight else 0.0,
    "unweighted_accuracy": sum(row["prediction"] == row["correct"] for row in predictions) / len(predictions),
    "unweighted_candidate_coverage": sum(row["correct_in_candidates"] for row in predictions) / len(predictions),
}

csv_path = OUTPUT_DIR / "q2_spell_correction_predictions.csv"
with csv_path.open("w", encoding="utf-8", newline="") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=[
        "correct", "error", "frequency", "prediction", "correct_in_candidates", "top_candidates"
    ])
    writer.writeheader()
    writer.writerows(predictions)
json_path = OUTPUT_DIR / "q2_spell_correction_results.json"
json_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2))
print(f"Saved predictions to {csv_path.name} and metrics to {json_path.name}.")


{
  "model": "argmax_correct_word(P(error|correct_word) * P(correct_word))",
  "channel": "P(error_piece|correct_piece) from count_1edit.txt with add-alpha smoothing and edit-character backoff, alpha=0.1",
  "candidate_generation": "dictionary words at exactly one Damerau edit",
  "evaluation_pairs": 39710,
  "distinct_errors": 36978,
  "frequency_weighted_examples": 41718,
  "frequency_weighted_accuracy": 0.29464499736324845,
  "candidate_coverage": 0.3671556642216789,
  "accuracy_when_gold_is_in_candidates": 0.802507018345629,
  "unweighted_accuracy": 0.28312767564845126,
  "unweighted_candidate_coverage": 0.34996222613951145
}
Saved predictions to q2_spell_correction_predictions.csv and metrics to q2_spell_correction_results.json.
